In [142]:
import csv
import io
import os
from pathlib import Path

import muon as mu
import numpy as np
import pandas as pd
import scipy.sparse as sp
from joblib import Parallel, delayed

In [147]:
# samples = [
#     ("mESC", "E7.5_rep2"),
#     ("mESC", "E8.0_rep1"),
#     ("mESC", "E8.0_rep2"),
#     ("mESC", "E8.5_rep1"),
#     ("mESC", "E8.5_rep2"),
#     ("kidney", "Ctrl_4weeks_1"),
#     ("kidney", "Ctrl_4weeks_2"),
#     ("kidney", "Ctrl_6months_1"),
#     ("GSE140203_shareseq", "skin_late_anagen"),
#     ("GSE140203_shareseq", "shareseq_brain"),
#     ("10x_E18_mouse_brain", "E18_brain"),
#     ("GSE246464_HSC", "young_rep1"),
#     ("GSE246464_HSC", "young_rep2"),
#     ("GSE246464_HSC", "old_rep1"),
#     ("GSE246464_HSC", "old_rep2")
# ]

# (tissue, sample). The tissue name is the folder under data/processed/<GENOME>/ and also
# selects <tissue>_label_map.tsv, the ground-truth folder <GENOME>/<tissue>/, and the output folder.
samples = [
    # ("ENCODE_adrenal", "adrenal_adult_ENCDO221IPH"),
    # ("ENCODE_bile_duct", "bile_duct_adult_ENCDO575WHY"),
    # ("ENCODE_colon", "colon_adult_ENCDO856ZOJ"),
    ("ENCODE_fallopian_tube", "fallopian_tube_adult_ENCDO411EVD"),
    ("ENCODE_heart", "heart_lv_adult_ENCDO575WHY"),
    ("ENCODE_lung", "lung_adult_ENCDO528BHB"),
    ("ENCODE_pancreas", "pancreas_adult_ENCDO221IPH"),
    ("GSE194122_bone_marrow", "s1d1"),
    ("GSE319044_lung_spleen_immune", "Spleen-SMO-04"),
    ("GSE207335_epidermis", "S1"),
    ("GSE240061_muscle_pre_exercise", "Ex-1_Pre_tr1"),
    ("GSE277326_retina_macula_rpe", "Retina_donor1"),
]

DATA_DIR = Path("/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data")
MULTIGRNTOOLS_RAW_DATA_DIR = Path(
    "/gpfs/Labs/Uzun/DATA/PROJECTS/"
    "2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS"
)
# Same mapping-file location as train_tf_to_tg_celltype_model.py.
CELL_TYPE_SPECIFIC_GT_DIR = DATA_DIR / "ground_truth_files" / "cell_type_specific"
# Ground-truth parquets are at CELL_TYPE_SPECIFIC_GT_DIR / GENOME / <tissue> / <slice>_ground_truth.parquet.
GENOME = "hg38"

# Each worker holds one whole sample in memory, so memory limits N_JOBS more than cores do.
N_JOBS = 3
# Polars writer threads per worker. Keep N_JOBS * POLARS_THREADS <= allocated cores.
POLARS_THREADS = 4
# Rows (genes or peaks) made dense at one time while writing.
ROWS_PER_CHUNK = 2000
# False: skip slices whose RNA and ATAC CSVs already exist, so an interrupted run resumes.
OVERWRITE = False

In [148]:
def csv_line(fields):
    buf = io.StringIO()
    csv.writer(buf, lineterminator="\n").writerow(fields)
    return buf.getvalue()


def write_feature_by_cell_csv(counts, cell_names, feature_names, csv_file):
    """Write a cell x feature count matrix as a feature x cell CSV.

    The output is byte-identical to
    pd.DataFrame(counts.toarray(), index=cell_names, columns=feature_names).T.to_csv(csv_file),
    but only ROWS_PER_CHUNK rows are dense at one time, and polars does the formatting.
    The file is written to a .tmp name and renamed at the end, so a killed worker
    never leaves a partial CSV that a later run would skip.
    """
    import polars as pl

    feature_by_cell = counts.T.tocsr() if sp.issparse(counts) else np.asarray(counts).T
    column_names = [f"c{i}" for i in range(len(cell_names))]
    tmp_file = csv_file.with_name(csv_file.name + ".tmp")

    with open(tmp_file, "wb") as f:
        f.write(csv_line([feature_names.name or "", *map(str, cell_names)]).encode())
        for start in range(0, feature_by_cell.shape[0], ROWS_PER_CHUNK):
            block = feature_by_cell[start:start + ROWS_PER_CHUNK]
            block = block.toarray() if sp.issparse(block) else block
            chunk = pl.DataFrame(block, schema=column_names, orient="row")
            chunk.insert_column(
                0,
                pl.Series("feature", np.asarray(feature_names[start:start + ROWS_PER_CHUNK], dtype=str)),
            )
            chunk.write_csv(f, include_header=False)

    tmp_file.replace(csv_file)


def align_rna_atac_counts(rna, atac):
    """Keep the cells present in both modalities, with ATAC put into RNA's cell order.

    intersect_obs keeps the shared cells but not a shared order (SHARE-seq skin stores
    ATAC cells in a different order than RNA). Rows are matched by cell name, so row i
    of both returned count matrices is the same cell.

    Returns (obs, rna_counts, atac_counts, n_rna_only, n_atac_only).
    """
    for name, adata in (("RNA", rna), ("ATAC", atac)):
        if not adata.obs_names.is_unique:
            raise ValueError(f"{name} has duplicated cell names; cannot align by name")

    rna_idx = np.flatnonzero(rna.obs_names.isin(atac.obs_names))
    shared_cells = rna.obs_names[rna_idx]
    atac_idx = atac.obs_names.get_indexer(shared_cells)

    obs = rna.obs.iloc[rna_idx]
    rna_counts = rna.layers["counts"][rna_idx]
    atac_counts = atac.layers["counts"][atac_idx]

    assert atac.obs_names[atac_idx].equals(obs.index)
    return obs, rna_counts, atac_counts, rna.n_obs - len(rna_idx), atac.n_obs - len(atac_idx)


def load_slice_members(tissue_name, celltype_counts):
    """Map each output slice to the annotated cell types it combines.

    Same logic as train_tf_to_tg_celltype_model.py: one slice per celltype_group.
    Slice names have spaces replaced by underscores, to match the ground-truth file
    names (e.g. kidney "B cells" -> B_cells_ground_truth.parquet).
    """
    label_map_path = CELL_TYPE_SPECIFIC_GT_DIR / f"{GENOME}_cell_type_maps" / f"{tissue_name}_label_map.tsv"

    if label_map_path.exists():
        label_map = pd.read_csv(label_map_path, sep="\t", comment="#")
        raw_slice_members = label_map.groupby("celltype_group")["celltype"].apply(list).to_dict()
        message = (
            f"{label_map_path.name}: {len(raw_slice_members)} slices from "
            f"{label_map['celltype'].nunique()} annotated cell types"
        )
    else:
        raw_slice_members = {celltype: [celltype] for celltype in celltype_counts.index}
        message = f"No {label_map_path.name}; one slice per annotated cell type ({len(raw_slice_members)})"

    # Names that differ only by spaces vs underscores go into one slice.
    slice_members = {}
    for slice_name, members in raw_slice_members.items():
        slice_members.setdefault(slice_name.replace(" ", "_"), []).extend(members)

    return slice_members, message


def export_sample(tissue_name, sample_name):
    """Write the RNA and ATAC count CSVs for every cell-type slice of one sample."""
    log = [f"=== {tissue_name} / {sample_name}"]

    h5mu_file = DATA_DIR / "processed" / GENOME / tissue_name / sample_name / "multiome_processed.h5mu"
    mdata = mu.read(h5mu_file)
    rna_all = mdata.mod["rna"]
    atac_all = mdata.mod["atac"]

    obs, rna_counts, atac_counts, n_rna_only, n_atac_only = align_rna_atac_counts(rna_all, atac_all)
    log.append(
        f"Aligned {len(obs):,} cells shared by RNA and ATAC "
        f"(dropped {n_rna_only:,} RNA-only, {n_atac_only:,} ATAC-only)"
    )

    celltype_counts = obs["celltype"].value_counts()
    slice_members, message = load_slice_members(tissue_name, celltype_counts)
    log.append(message)

    gt_dir = CELL_TYPE_SPECIFIC_GT_DIR / GENOME / tissue_name

    for slice_name, members in slice_members.items():
        # Mapping files may contain cell types that are absent from this sample.
        present_members = [celltype for celltype in members if celltype in celltype_counts.index]
        if not present_members:
            log.append(f"  Skipping {slice_name}: none of its mapped cell types occur in {sample_name}")
            continue

        # A slice without ground truth cannot be scored, so the other methods need not run on it.
        if not (gt_dir / f"{slice_name}_ground_truth.parquet").exists():
            n_cells = int(celltype_counts[present_members].sum())
            log.append(f"  Skipping {slice_name}: no ground truth file in {gt_dir} ({n_cells:,} cells)")
            continue

        output_dir = MULTIGRNTOOLS_RAW_DATA_DIR / GENOME / tissue_name / sample_name / slice_name
        output_dir.mkdir(parents=True, exist_ok=True)
        
        rna_csv_file = output_dir / f"{slice_name}_RNA.csv"
        atac_csv_file = output_dir / f"{slice_name}_ATAC.csv"

        if not OVERWRITE and rna_csv_file.exists() and atac_csv_file.exists():
            log.append(f"  Skipping {slice_name}: CSVs already exist")
            continue

        # Combine all cells assigned to this mapped slice/lineage.
        # The aligned matrices share one cell order, so one mask slices both.
        slice_mask = obs["celltype"].isin(present_members).to_numpy()
        cell_names = obs.index[slice_mask]

        write_feature_by_cell_csv(
            rna_counts[slice_mask], cell_names, rna_all.var_names, rna_csv_file
        )
        write_feature_by_cell_csv(
            atac_counts[slice_mask], cell_names, atac_all.var_names, atac_csv_file
        )

        log.append(
            f"  {slice_name}: {len(cell_names):,} cells, {rna_all.n_vars:,} genes, "
            f"{atac_all.n_vars:,} peaks; cell types {present_members}"
        )
        log.append(f"    -> {output_dir}")

    return log


# Workers inherit this at start-up. Without it, each worker's polars would use every core.
os.environ["POLARS_MAX_THREADS"] = str(POLARS_THREADS)

logs = Parallel(n_jobs=min(N_JOBS, len(samples)), backend="loky", verbose=10)(
    delayed(export_sample)(tissue_name, sample_name)
    for tissue_name, sample_name in samples
)

for log in logs:
    print("\n".join(log))

[Parallel(n_jobs=3)]: Using backend LokyBackend with 3 concurrent workers.
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/muon/_core/preproc.py:32: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/muon/_core/preproc.py:32: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/muon/_core/preproc.py:32: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  if Version(scanpy.__version__) < Version("1.10"):
/gpfs/Home/esm5360/miniconda3/envs/my_env/lib/python3.10/site-packages/mudata/_core/mudata.py:1403: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by

=== ENCODE_fallopian_tube / fallopian_tube_adult_ENCDO411EVD
Aligned 5,115 cells shared by RNA and ATAC (dropped 0 RNA-only, 0 ATAC-only)
ENCODE_fallopian_tube_label_map.tsv: 5 slices from 6 annotated cell types
  Skipping Endothelial_cells: CSVs already exist
  Skipping Fallopian_tube_epithelium: CSVs already exist
  Skipping Fibroblasts: CSVs already exist
  Macrophages: 68 cells, 27,088 genes, 121,530 peaks; cell types ['Macrophage']
    -> /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/hg38/ENCODE_fallopian_tube/fallopian_tube_adult_ENCDO411EVD/Macrophages
  Smooth_muscle_cells: 150 cells, 27,088 genes, 121,530 peaks; cell types ['Smooth muscle']
    -> /gpfs/Labs/Uzun/DATA/PROJECTS/2024.GRN_BENCHMARKING.MOELLER/MUON_FILTERED_COUNT_DATASETS/hg38/ENCODE_fallopian_tube/fallopian_tube_adult_ENCDO411EVD/Smooth_muscle_cells
=== ENCODE_heart / heart_lv_adult_ENCDO575WHY
Aligned 9,502 cells shared by RNA and ATAC (dropped 0 RNA-only, 0 ATAC-only)


[Parallel(n_jobs=3)]: Done   9 out of   9 | elapsed:  3.7min finished
